## 7. Magentic Orchestration - Manager-Led Collaboration

**Goal**: Watch an LLM-powered manager plan a task, choose specialist agents, check progress, and synthesize an answer using `MagenticBuilder`.

**Scenario**: Design a two-hour beginner workshop within a fixed budget.

| Agent | Responsibility |
|---|---|
| Manager | Plans, delegates, tracks completion, and produces the final answer |
| Planner | Drafts the agenda and budget |
| Reviewer | Checks timing, arithmetic, and requirements; suggests corrections |

Unlike a sequential workflow, there is no fixed Planner-to-Reviewer edge: the manager decides who speaks next and can ask for revisions. The framework maintains a **task ledger** (facts and plan) and a **progress ledger** (completion, next speaker, and instructions).

- Planner: Creates practical workshop agendas and itemized budgets.
- Reviewer: Independently checks agenda durations, budget arithmetic, and constraints.

In [1]:
# Setup
import os
import sys

sys.path.insert(0, os.path.abspath("..") if "notebooks" in os.getcwd() else ".")

from agent_framework import Agent, AgentResponse, AgentResponseUpdate, Message
from agent_framework.orchestrations import (
    MagenticBuilder,
    MagenticOrchestratorEvent,
    MagenticProgressLedger,
)
from utils.config import get_env, get_foundry_client

client = get_foundry_client()
print(f"Project: {get_env('AZURE_AI_PROJECT_ENDPOINT_FULL')}")
print(f"Model:   {get_env('AZURE_AI_MODEL_DEPLOYMENT_NAME')}")

c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_skills.py:116: ExperimentalWarning: [SKILLS] SkillResource is experimental and may change or be removed in future versions without notice.
c:\Dibyendu\Code\1FinalSamples\agent-framework-demo-py\.venv\Lib\site-packages\agent_framework\_harness\_memory.py:651: ExperimentalWarning: [HARNESS] MemoryStore is experimental and may change or be removed in future versions without notice.


2026-10-26 11:44:53,854 - azure.ai.agentserver - INFO - Application Insights not configured, telemetry export disabled.
Project: https://foundry-demo-us-east-resource.services.ai.azure.com/api/projects/prj-foundry-demo-us-east-dev-001
Model:   gpt-4o


---
### Step 1 - Define the team

All three agents use the same Foundry client but have different responsibilities. The manager uses the framework's built-in planning and progress prompts; no custom manager implementation is needed.

In [2]:
manager = Agent(
    name="Manager",
    description="Coordinates planning, independent review, and final synthesis.",
    instructions=(
        "Coordinate this short demo efficiently. Ask Planner for an initial draft, "
        "then ask Reviewer to check it. Do not consider the task complete until "
        "both specialists have contributed. Request corrections only if needed. "
        "Synthesize a concise final answer using the reviewed work. "
        "Follow the framework's requested response format for each planning step."
    ),
    client=client,
)

planner = Agent(
    name="Planner",
    description="Creates practical workshop agendas and itemized budgets.",
    instructions=(
        "Draft or revise a concise workshop plan using only the supplied facts. "
        "Include timed agenda items, an itemized budget, the total, and remaining funds. "
        "Respect all constraints. Do not ask the user questions or claim to browse the web."
    ),
    client=client,
)

reviewer = Agent(
    name="Reviewer",
    description="Independently checks agenda durations, budget arithmetic, and constraints.",
    instructions=(
        "Review the proposed workshop against the user's requirements. "
        "Explicitly check the sum of agenda minutes, all cost calculations, "
        "the budget ceiling, and beginner suitability. Report concrete corrections "
        "if needed; otherwise state that the plan passes. Keep your review concise."
    ),
    client=client,
)

print("Team ready: Manager, Planner, Reviewer")

Team ready: Manager, Planner, Reviewer


---
### Step 2 - Build the Magentic workflow

Participant responses are exposed as `intermediate` events; the manager's final response is an `output` event. The manager also emits planning and progress events, which the next cell prints.

The six-round and two-reset limits bound coordination, not the exact number of model calls. If a limit is reached before completion, the run cell reports an error rather than presenting an incomplete run as successful.

In [3]:
workflow = MagenticBuilder(
    participants=[planner, reviewer],
    manager_agent=manager,
    intermediate_output_from=[planner, reviewer],
    max_round_count=6,
    max_stall_count=2,
    max_reset_count=2,
).build()

print("Magentic workflow built. The manager chooses the next specialist.")

Magentic workflow built. The manager chooses the next specialist.


---
### Step 3 - Run the demo and watch the decisions

Expected flow: **manager plan -> Planner draft -> Reviewer checks -> manager final answer**. The actual routing is model-driven and can include a revision.

For a fresh demonstration, rerun from **Step 1** to create new agents and a new workflow. Edit `task` below to try a different planning problem. No interactive inputs are required.

In [4]:
task = """Plan a beginner-friendly, two-hour Python workshop for 20 people.
Use these fixed demo costs: materials $5 per person, snacks $8 per person,
instructor $200 flat, and venue free. The total budget is $500.
Include a hands-on exercise. Ask Planner to draft and Reviewer to independently
verify the plan before finishing. Return a concise agenda totaling exactly
120 minutes, an itemized budget, total cost, and remaining funds.
Use only these facts; no external research is required."""

print(f"TASK:\n{task}\n")
last_section = None
final_answer_parts = []
request_satisfied = False

async for event in workflow.run(task, stream=True):
    if event.type == "magentic_orchestrator" and isinstance(event.data, MagenticOrchestratorEvent):
        content = event.data.content
        print(f"\n\n[Manager: {event.data.event_type.value}]")
        if isinstance(content, Message):
            print(content.text)
        elif isinstance(content, MagenticProgressLedger):
            request_satisfied = content.is_request_satisfied.answer is True
            print(f"Task complete: {request_satisfied}")
            if not request_satisfied:
                print(f"Next agent: {content.next_speaker.answer}")
                print(f"Instruction: {content.instruction_or_question.answer}")
        last_section = None

    elif event.type in ("intermediate", "output") and isinstance(
        event.data, (AgentResponse, AgentResponseUpdate)
    ):
        section = (event.type, event.executor_id)
        if section != last_section:
            label = "Manager output" if event.type == "output" else event.executor_id
            print(f"\n\n--- {label} ---\n")
            last_section = section
        print(event.data.text, end="", flush=True)
        if event.type == "output":
            final_answer_parts.append(event.data.text)

final_answer = "".join(final_answer_parts).strip()
print()
if not request_satisfied or not final_answer:
    raise RuntimeError(
        "The workflow stopped without a completed answer. Inspect the events above; "
        "simplify the task or adjust the round/reset limits, then rerun from Step 1."
    )
print("\nCompleted: the manager synthesized the reviewed plan.")

TASK:
Plan a beginner-friendly, two-hour Python workshop for 20 people.
Use these fixed demo costs: materials $5 per person, snacks $8 per person,
instructor $200 flat, and venue free. The total budget is $500.
Include a hands-on exercise. Ask Planner to draft and Reviewer to independently
verify the plan before finishing. Return a concise agenda totaling exactly
120 minutes, an itemized budget, total cost, and remaining funds.
Use only these facts; no external research is required.



[Manager: plan_created]

We are working to address the following user request:

Plan a beginner-friendly, two-hour Python workshop for 20 people.
Use these fixed demo costs: materials $5 per person, snacks $8 per person,
instructor $200 flat, and venue free. The total budget is $500.
Include a hands-on exercise. Ask Planner to draft and Reviewer to independently
verify the plan before finishing. Return a concise agenda totaling exactly
120 minutes, an itemized budget, total cost, and remaining funds.
Use

---
### What to point out during the demo

- **Plan**: The manager establishes facts and a strategy before delegating.
- **Dynamic routing**: Progress events show the next agent and its specific assignment.
- **Review**: The Reviewer checks the Planner's work instead of independently answering the original question.
- **Synthesis**: The manager produces the final answer, rather than returning only the last participant response.
- **Sanity check**: The supplied costs total `$460`, leaving `$40`; agenda durations should total `120` minutes. Wording and agenda items vary between runs.
- **Replanning**: Magentic can revise its plan when progress stalls. A straightforward task like this normally finishes without needing a reset.

This is a local Agent Framework workflow calling an existing Foundry model, not a deployed Foundry-hosted workflow. It does not save checkpoints or register persistent agents.